# Khai thác tập phổ biến bằng Eclat (Julia) — Demo & Tái lập trên Google Colab

Đồ án môn **Khai thác dữ liệu và ứng dụng (HCMUS)**.
- **Thuật toán:** Eclat (Zaki, *IEEE TKDE* 2000)
- **Thành viên:** Phạm Trần Anh Quân (A — Thuật toán & Hiệu năng), Nguyễn Tiến Cường (B — Kiểm chứng, Thực nghiệm & Dữ liệu)
- **Hai chế độ chạy:** `CHE_DO = "nhanh"` (≤ 40 phút trên Colab miễn phí) hoặc `CHE_DO = "day_du"`.

Mỗi phần dưới đây được thiết kế **độc lập**, tự kiểm tra điều kiện cần trước khi chạy.

In [ ]:
# === Cấu hình chung ===
import os
CHE_DO = "nhanh"      # "nhanh" hoặc "day_du"
REPO_URL = "https://github.com/24127226/16-FIM-DATA-MINING.git"
REPO_BRANCH = "main"  # sẽ đổi sang tag "v1.0" ở bản nộp cuối (D19)
REPO_DIR = "/content/16-FIM-DATA-MINING"
JULIA_VERSION = "1.11.4"
os.environ["FIM_DATA_BASE"] = "https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main"

print(f"Chế độ: {CHE_DO} | Repo: {REPO_URL} ({REPO_BRANCH})")

## Phần 1. Cài đặt môi trường (Julia 1.11.4, Java, SPMF)

Clone repo, cài đúng **Julia 1.11.4**, cài gói Julia (`Pkg.instantiate()`), kiểm tra Java và tải `spmf.jar`.

In [ ]:
import os, shutil, subprocess

# 1. Clone repo nếu chưa có
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# 2. Cài Julia 1.11.4 nếu chưa có
julia_bin = f"/usr/local/julia-{JULIA_VERSION}/bin/julia"
if not os.path.exists(julia_bin) and shutil.which("julia") is None:
    url = f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
    subprocess.run(f"curl -fsSL {url} | tar -xz -C /usr/local", shell=True, check=True)
    subprocess.run(["ln", "-sf", julia_bin, "/usr/local/bin/julia"], check=True)

!julia --version
!julia --project=. -e "using Pkg; Pkg.instantiate(); Pkg.precompile()"

In [ ]:
# 3. Kiểm tra Java và tải spmf.jar (SPMF bản mới yêu cầu Java >= 25 để đối chiếu)
!java -version || true
!julia --project=. scripts/fetch_spmf.jl
!julia --project=. scripts/env_info.jl

## Phần 2. Tải dữ liệu và kiểm tra SHA-256

Tải các bộ benchmark (`chess`, `mushroom`, `retail`, `T10I4D100K`) và kiểm tra toàn vẹn SHA-256. Không yêu cầu đăng nhập.

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl benchmark

## Phần 3. Demo CLI trên dữ liệu toy

Chạy nguyên văn lệnh mẫu của đề trên `pasquier99.txt` (40% → 15 tập) và `bookstore.txt` (0.5 → 19 tập).

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. demo.jl --minsup 40% -o - data/toy/pasquier99.txt
!julia --project=. demo.jl --minsup 0.5 --stats -o output_bookstore.txt data/toy/bookstore.txt

## Phần 4. Chạy bộ kiểm thử tự động (`test/runtests.jl`)

Gồm kiểm thử I/O, `minsup_count`, oracle duyệt trâu trên 200 CSDL ngẫu nhiên, ví dụ chuẩn, các ca biên và kiểm thử benchmark.

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. test/runtests.jl

## Phần 5. Chương 2 — Thực nghiệm & Đánh giá hiệu năng

Chạy lại thực nghiệm theo chế độ `CHE_DO` (`nhanh` / `day_du`) bằng `scripts/bench.jl` và vẽ lại hình từ CSV.

In [ ]:
import os
os.chdir(REPO_DIR)
mode_arg = "quick" if CHE_DO == "nhanh" else "full"
repeats = 1 if CHE_DO == "nhanh" else 3
!julia --project=. scripts/bench.jl --mode {mode_arg} --variants v0,v1 --repeats {repeats} --out results/bench_colab.csv
!head -n 20 results/bench_colab.csv

## Phần 6. Chương 3 — Ứng dụng 1: Phân tích giỏ hàng (Groceries)

Tải `groceries.csv` (9.835 giao dịch, 169 mặt hàng), biến đổi dữ liệu theo 2 phương án (B08), chạy `EclatFIM` khai thác tập phổ biến và luật kết hợp, xuất top-10 theo `lift` (B09) và kiểm tra độ ổn định chia 80/20 (B10).

In [ ]:
import os
from IPython.display import Markdown, display

os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl groceries
!julia --project=. apps/groceries/run_groceries.jl

with open("results/groceries_summary.md", "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

## Phần 7. Chương 3 — Ứng dụng 2: Mẫu trực quan bằng Bag of Visual Words (Caltech-101)

*(Phần của thành viên A — Quân, sẽ bổ sung ở Tuần 5–6.)*

## Phần 8. Tổng kết & Báo cáo

Báo cáo đầy đủ đặt tại `docs/Report.pdf` trong repository.